# Google Trends event-word pull pipeline

This notebook uses manually curated `TOP_QUERIES_RAW` and `RISING_QUERIES_RAW`.

Design:
- Keep the query text exactly as written for Google Trends.
- Deduplicate exact duplicate query strings across top/rising so each query is pulled once.
- Preserve source metadata: `top`, `rising`, or `top;rising`.
- Use only folder-name sanitization for Windows-safe query folders.
- Save each query into its own folder with `chunks/`, `diagnostics/`, `stitched/`, and `metadata.json`.


In [ ]:
from pathlib import Path
import json
import random
import re
import time
from datetime import datetime, timedelta
from collections import defaultdict

import pandas as pd

try:
    from pytrends.request import TrendReq
except ImportError as e:
    raise ImportError(
        "pytrends is required. Install it with: pip install pytrends"
    ) from e


In [6]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path(r"C:/Python/CSUREMM")
EVENT_WORD_DIR = PROJECT_DIR / "event_words"
BASE_DIR = PROJECT_DIR / "data_events"

EVENT_WORD_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR.mkdir(parents=True, exist_ok=True)

# ── Google Trends parameters ──────────────────────────────────────────────────
GEO = "US"
START_DATE = "2022-01-01"
END_DATE = "2026-05-31"

# 270-day chunks work well for daily Google Trends pulls.
CHUNK_DAYS = 270

# Overlap helps stitching adjacent chunks.
OVERLAP_DAYS = 30

# Sleep settings: conservative randomized pacing.
STANDARD_SLEEP_RANGE = (12.0, 24.0)
MICRO_NAP_EVERY = 5
MICRO_NAP_RANGE = (45.0, 90.0)

MAX_RETRIES = 5
RETRY_SLEEP_RANGE = (60.0, 300.0)


In [7]:
# ── Hardcoded query lists ─────────────────────────────────────────────────────
# These are manually curated inputs. The notebook does not clean or normalize them.

TOP_QUERIES_RAW = r"""
"""

RISING_QUERIES_RAW = r"""when we were young festival
aaron judge
canelo vs bivol
ime udoka
kohls
obi wan kenobi
southwest airlines
unblocked games
2024 united states presidential election
australian open 2022
facebook login
iran
kansas city chiefs
"""

In [8]:
# ── Minimal parsing, deduplication, and folder-name sanitization ──────────────

def parse_query_block(raw: str) -> list[str]:
    """Split a newline-delimited query block into non-empty strings."""
    return [
        line.strip()
        for line in raw.splitlines()
        if line.strip()
    ]


def folder_name(query: str) -> str:
    """
    Create a Windows-safe folder name.

    Important:
    - This does NOT modify the query sent to Google Trends.
    - It only creates a filesystem-safe folder slug.
    """
    name = query.strip().lower()
    name = re.sub(r'[<>:"/\\|?*]+', "", name)
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("._ ")

    if not name:
        name = "query"

    # Windows path-length protection.
    return name[:120]


def make_unique_folder_slug(query: str, used: set[str]) -> str:
    """Create a unique folder slug, adding suffixes only if needed."""
    base = folder_name(query)
    slug = base
    i = 2

    while slug in used:
        slug = f"{base}_{i}"
        i += 1

    used.add(slug)
    return slug


top_queries = parse_query_block(TOP_QUERIES_RAW)
rising_queries = parse_query_block(RISING_QUERIES_RAW)

# Track exact duplicates within each list.
top_duplicate_rows = (
    pd.Series(top_queries)
    .value_counts()
    .loc[lambda s: s.gt(1)]
    .rename_axis("query")
    .reset_index(name="count")
)

rising_duplicate_rows = (
    pd.Series(rising_queries)
    .value_counts()
    .loc[lambda s: s.gt(1)]
    .rename_axis("query")
    .reset_index(name="count")
)

# Build exact-query source metadata.
sources_by_query = defaultdict(set)

for q in top_queries:
    sources_by_query[q].add("top")

for q in rising_queries:
    sources_by_query[q].add("rising")

# Preserve order: all top queries first, then rising-only queries.
ordered_queries = []
seen = set()

for q in top_queries + rising_queries:
    if q not in seen:
        seen.add(q)
        ordered_queries.append(q)

used_slugs = set()
metadata_rows = []

for q in ordered_queries:
    sources = sorted(sources_by_query[q])
    metadata_rows.append({
        "query": q,
        "folder": make_unique_folder_slug(q, used_slugs),
        "source": ";".join(sources),
        "in_top": "top" in sources,
        "in_rising": "rising" in sources,
    })

query_metadata = pd.DataFrame(metadata_rows)

cross_source_duplicates = (
    query_metadata
    .loc[query_metadata["in_top"] & query_metadata["in_rising"], ["query", "source"]]
    .reset_index(drop=True)
)

query_metadata.to_csv(BASE_DIR / "query_metadata.csv", index=False)
top_duplicate_rows.to_csv(BASE_DIR / "duplicates_within_top.csv", index=False)
rising_duplicate_rows.to_csv(BASE_DIR / "duplicates_within_rising.csv", index=False)
cross_source_duplicates.to_csv(BASE_DIR / "duplicates_across_top_rising.csv", index=False)

print(f"Raw top queries:      {len(top_queries)}")
print(f"Raw rising queries:   {len(rising_queries)}")
print(f"Unique queries pulled:{len(query_metadata)}")
print(f"Within-top duplicates removed:    {len(top_duplicate_rows)}")
print(f"Within-rising duplicates removed: {len(rising_duplicate_rows)}")
print(f"Cross-source duplicates pulled once: {len(cross_source_duplicates)}")

display(query_metadata.head())
display(cross_source_duplicates.head(20))


Raw top queries:      0
Raw rising queries:   13
Unique queries pulled:13
Within-top duplicates removed:    0
Within-rising duplicates removed: 0
Cross-source duplicates pulled once: 0


,query,folder,source,in_top,in_rising
0,when we were young festival,when_we_were_young_festival,rising,False,True
1,aaron judge,aaron_judge,rising,False,True
2,canelo vs bivol,canelo_vs_bivol,rising,False,True
3,ime udoka,ime_udoka,rising,False,True
4,kohls,kohls,rising,False,True


,query,source


## Deduplication note

The notebook removes only **exact duplicate query strings after stripping outer whitespace**. It does not merge near-duplicates such as `chat gpt` vs `chatgpt`, `s&p 500` vs `sp500`, or curly-apostrophe variants vs straight-apostrophe variants.

Cross-source duplicates are not discarded conceptually: they are pulled once and marked as `source = "rising;top"` in `query_metadata.csv`.


In [9]:
# ── Chunk construction ───────────────────────────────────────────────────────

def make_time_windows(
    start_date: str,
    end_date: str,
    chunk_days: int = 270,
    overlap_days: int = 30,
) -> list[tuple[str, str]]:
    """
    Create overlapping date windows formatted for pytrends timeframe strings.

    Google Trends timeframe end date is inclusive in the string representation.
    """
    start = pd.to_datetime(start_date).date()
    end = pd.to_datetime(end_date).date()

    windows = []
    cur = start

    while cur <= end:
        chunk_end = min(cur + timedelta(days=chunk_days - 1), end)
        windows.append((cur.isoformat(), chunk_end.isoformat()))

        if chunk_end >= end:
            break

        cur = chunk_end - timedelta(days=overlap_days - 1)

    return windows


TIME_WINDOWS = make_time_windows(
    START_DATE,
    END_DATE,
    chunk_days=CHUNK_DAYS,
    overlap_days=OVERLAP_DAYS,
)

TIME_WINDOWS


[('2022-01-01', '2022-09-27'),
 ('2022-08-29', '2023-05-25'),
 ('2023-04-26', '2024-01-20'),
 ('2023-12-22', '2024-09-16'),
 ('2024-08-18', '2025-05-14'),
 ('2025-04-15', '2026-01-09'),
 ('2025-12-11', '2026-05-31')]

In [10]:
# ── Pull helpers ──────────────────────────────────────────────────────────────

def standard_pause(index: int) -> None:
    """Conservative randomized pacing between successful requests."""
    if index > 0 and index % MICRO_NAP_EVERY == 0:
        sleep_time = random.uniform(*MICRO_NAP_RANGE)
        print(f"Micro-nap after {index} requests: {sleep_time:.1f}s")
    else:
        sleep_time = random.uniform(*STANDARD_SLEEP_RANGE)
        print(f"Standard pause: {sleep_time:.1f}s")

    time.sleep(sleep_time)


def build_pytrends() -> TrendReq:
    """
    Create a fresh pytrends object.

    Recreating this periodically is more robust than keeping one long-lived object.
    """
    return TrendReq(
        hl="en-US",
        tz=360,
        timeout=(10, 25),
        retries=0,
        backoff_factor=0,
    )


def pull_one_chunk(
    pytrends: TrendReq,
    query: str,
    start: str,
    end: str,
    geo: str = "US",
    max_retries: int = MAX_RETRIES,
) -> pd.DataFrame:
    """Pull one query over one time window with conservative retry behavior."""
    timeframe = f"{start} {end}"

    for attempt in range(max_retries):
        try:
            pytrends.build_payload(
                kw_list=[query],
                timeframe=timeframe,
                geo=geo,
            )

            df = pytrends.interest_over_time()

            if df is None or df.empty:
                return pd.DataFrame(columns=["Time", query])

            df = df.drop(columns=["isPartial"], errors="ignore")
            df = df.reset_index().rename(columns={"date": "Time"})

            return df

        except Exception as e:
            msg = str(e).lower()
            is_rate_limited = (
                "429" in msg
                or "too many requests" in msg
                or "rate" in msg
                or "quota" in msg
            )

            if attempt == max_retries - 1:
                raise

            if is_rate_limited:
                sleep_time = random.uniform(*RETRY_SLEEP_RANGE)
                print(
                    f"Rate-limit-like error for {query!r}, "
                    f"{start} to {end}, attempt {attempt + 1}/{max_retries}. "
                    f"Sleeping {sleep_time:.1f}s."
                )
                time.sleep(sleep_time)
                pytrends = build_pytrends()
            else:
                sleep_time = random.uniform(20.0, 60.0)
                print(
                    f"Request error for {query!r}, "
                    f"{start} to {end}, attempt {attempt + 1}/{max_retries}: {e}. "
                    f"Sleeping {sleep_time:.1f}s."
                )
                time.sleep(sleep_time)

    raise RuntimeError(f"Failed to pull {query!r} for {start} to {end}.")


In [11]:
# ── Stitching and diagnostics ────────────────────────────────────────────────

def read_chunk_file(path: Path, query: str) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["Time"])

    value_cols = [c for c in df.columns if c != "Time"]
    if not value_cols:
        return pd.DataFrame(columns=["Time", query])

    value_col = value_cols[0]

    return (
        df[["Time", value_col]]
        .rename(columns={value_col: query})
        .sort_values("Time")
        .drop_duplicates("Time")
    )


def stitch_chunks(chunk_files: list[Path], query: str) -> pd.DataFrame:
    """
    Simple overlap-aware stitching.

    The current implementation averages duplicate dates across overlapping chunks.
    This is conservative and transparent. If you later want ratio-rescaling,
    this function is the place to replace the stitching method.
    """
    if not chunk_files:
        return pd.DataFrame(columns=["Time", query])

    pieces = [
        read_chunk_file(path, query)
        for path in sorted(chunk_files)
    ]

    long = pd.concat(pieces, ignore_index=True)
    long[query] = pd.to_numeric(long[query], errors="coerce")

    stitched = (
        long
        .groupby("Time", as_index=False)[query]
        .mean()
        .sort_values("Time")
    )

    return stitched


def make_chunk_diagnostics(
    query: str,
    chunk_files: list[Path],
) -> pd.DataFrame:
    rows = []

    for path in sorted(chunk_files):
        try:
            df = read_chunk_file(path, query)
            rows.append({
                "file": path.name,
                "n_rows": len(df),
                "min_date": df["Time"].min() if not df.empty else pd.NaT,
                "max_date": df["Time"].max() if not df.empty else pd.NaT,
                "missing_values": df[query].isna().sum() if query in df else None,
            })
        except Exception as e:
            rows.append({
                "file": path.name,
                "n_rows": None,
                "min_date": pd.NaT,
                "max_date": pd.NaT,
                "missing_values": None,
                "error": str(e),
            })

    return pd.DataFrame(rows)


def make_stitching_diagnostics(stitched: pd.DataFrame, query: str) -> pd.DataFrame:
    if stitched.empty:
        return pd.DataFrame([{
            "query": query,
            "n_rows": 0,
            "min_date": pd.NaT,
            "max_date": pd.NaT,
            "expected_days": 0,
            "observed_days": 0,
            "missing_days": None,
        }])

    expected = pd.date_range(START_DATE, END_DATE, freq="D")
    observed = pd.DatetimeIndex(stitched["Time"])

    return pd.DataFrame([{
        "query": query,
        "n_rows": len(stitched),
        "min_date": stitched["Time"].min(),
        "max_date": stitched["Time"].max(),
        "expected_days": len(expected),
        "observed_days": observed.nunique(),
        "missing_days": len(expected.difference(observed)),
    }])


In [12]:
# ── Main query runner ────────────────────────────────────────────────────────

def write_query_metadata(row: pd.Series, query_dir: Path) -> None:
    meta = {
        "query": row["query"],
        "folder": row["folder"],
        "source": row["source"],
        "in_top": bool(row["in_top"]),
        "in_rising": bool(row["in_rising"]),
        "geo": GEO,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "chunk_days": CHUNK_DAYS,
        "overlap_days": OVERLAP_DAYS,
        "created_at": datetime.now().isoformat(timespec="seconds"),
    }

    with open(query_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)


def run_one_query(row: pd.Series, request_counter_start: int = 0) -> int:
    query = row["query"]
    folder = row["folder"]

    query_dir = BASE_DIR / folder
    chunks_dir = query_dir / "chunks"
    diagnostics_dir = query_dir / "diagnostics"
    stitched_dir = query_dir / "stitched"

    chunks_dir.mkdir(parents=True, exist_ok=True)
    diagnostics_dir.mkdir(parents=True, exist_ok=True)
    stitched_dir.mkdir(parents=True, exist_ok=True)

    write_query_metadata(row, query_dir)

    pytrends = build_pytrends()
    request_counter = request_counter_start

    chunk_files = []

    for start, end in TIME_WINDOWS:
        chunk_path = chunks_dir / f"gt_{folder}_{start}_{end}.csv"
        chunk_files.append(chunk_path)

        if chunk_path.exists():
            print(f"Exists, skipping: {chunk_path.name}")
            continue

        print(f"Pulling {query!r}: {start} to {end}")

        df = pull_one_chunk(
            pytrends=pytrends,
            query=query,
            start=start,
            end=end,
            geo=GEO,
        )

        df.to_csv(chunk_path, index=False)

        request_counter += 1
        standard_pause(request_counter)

    existing_chunk_files = [
        path for path in chunk_files
        if path.exists()
    ]

    chunk_diag = make_chunk_diagnostics(query, existing_chunk_files)
    chunk_diag.to_csv(
        diagnostics_dir / "chunk_diagnostics.csv",
        index=False,
    )

    stitched = stitch_chunks(existing_chunk_files, query)
    stitched_path = stitched_dir / f"gt_stitched_{folder}_{START_DATE}_{END_DATE}.csv"
    stitched.to_csv(stitched_path, index=False)

    stitch_diag = make_stitching_diagnostics(stitched, query)
    stitch_diag.to_csv(
        diagnostics_dir / "stitching_diagnostics.csv",
        index=False,
    )

    return request_counter


def run_all_queries(start_at: int = 0, stop_after: int | None = None) -> None:
    """
    Pull all queries starting at metadata row index `start_at`.

    Use stop_after for testing, e.g. stop_after=2.
    """
    run_log_path = BASE_DIR / "run_log.csv"

    request_counter = 0
    rows = query_metadata.iloc[start_at:]

    if stop_after is not None:
        rows = rows.iloc[:stop_after]

    log_rows = []

    for idx, row in rows.iterrows():
        query = row["query"]
        folder = row["folder"]

        print("=" * 80)
        print(f"[{idx}] {query!r} -> {folder}")

        try:
            request_counter = run_one_query(
                row=row,
                request_counter_start=request_counter,
            )

            status = "done"
            error = ""

        except Exception as e:
            status = "failed"
            error = str(e)
            print(f"FAILED: {query!r}: {error}")

        log_rows.append({
            "index": idx,
            "query": query,
            "folder": folder,
            "source": row["source"],
            "status": status,
            "error": error,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        })

        pd.DataFrame(log_rows).to_csv(run_log_path, index=False)

    print("Finished run.")


In [13]:
# ── Test run ─────────────────────────────────────────────────────────────────
# Start with 1 or 2 queries to verify file structure and Google Trends access.
# Then set stop_after=None to run the full list.

run_all_queries(
    start_at=0,
    stop_after=None,
)


[0] 'when we were young festival' -> when_we_were_young_festival
Pulling 'when we were young festival': 2022-01-01 to 2022-09-27
Standard pause: 15.7s
Pulling 'when we were young festival': 2022-08-29 to 2023-05-25


KeyboardInterrupt: 

In [ ]:
# ── Full run ─────────────────────────────────────────────────────────────────
# Uncomment after the test run works.

# run_all_queries(
#     start_at=0,
#     stop_after=None,
# )


## Expected file structure

```text
C:/Python/CSUREMM/data_events/
    query_metadata.csv
    duplicates_within_top.csv
    duplicates_within_rising.csv
    duplicates_across_top_rising.csv
    run_log.csv

    weather/
        metadata.json
        chunks/
            gt_weather_2022-01-01_2022-09-27.csv
            ...
        diagnostics/
            chunk_diagnostics.csv
            stitching_diagnostics.csv
        stitched/
            gt_stitched_weather_2022-01-01_2026-05-31.csv
```

Each query has exactly one canonical folder, even if it appears in both top and rising lists.


## Section: fixing stitch: gt_fix

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


DATA_DIR = Path(r"C:/Python/CSUREMM/data_events")
START_DATE = "2022-01-01"
END_DATE = "2026-05-31"


def read_chunk_file(path: Path, query: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        parse_dates=["Time"]
    )

    value_cols = [
        c for c in df.columns
        if c != "Time"
    ]

    if len(value_cols) != 1:
        raise ValueError(
            f"{path.name} has {len(value_cols)} value columns: {value_cols}"
        )

    value_col = value_cols[0]

    df = (
        df[["Time", value_col]]
        .rename(columns={value_col: query})
        .sort_values("Time")
        .drop_duplicates("Time", keep="first")
        .reset_index(drop=True)
    )

    df[query] = pd.to_numeric(
        df[query],
        errors="coerce"
    )

    return df


def estimate_overlap_scale(
    base: pd.DataFrame,
    new: pd.DataFrame,
    query: str,
    min_overlap_days: int = 10,
) -> dict:

    overlap = pd.merge(
        base[["Time", query]].rename(columns={query: "base_value"}),
        new[["Time", query]].rename(columns={query: "new_value"}),
        on="Time",
        how="inner"
    ).dropna(
        subset=["base_value", "new_value"]
    )

    valid = overlap[
        (overlap["base_value"] > 0) &
        (overlap["new_value"] > 0)
    ].copy()

    # exclude spike days from the RATIO estimate only, using the base series'
    # own distribution as the spike threshold
    thresh = valid["base_value"].quantile(spike_pctile / 100)
    calm = valid[valid["base_value"] <= thresh]
    if len(calm) >= min_overlap_days:
        valid = calm  # prefer spike-excluded overlap when enough calm days exist

    if len(overlap) >= 2 and overlap["base_value"].std() > 0 and overlap["new_value"].std() > 0:
        corr = overlap["base_value"].corr(overlap["new_value"])
    else:
        corr = np.nan

    if len(valid) >= min_overlap_days:
        ratio_new_over_base = (
            valid["new_value"] /
            valid["base_value"]
        )

        median_ratio = ratio_new_over_base.median()

        scale_new_to_base = (
            1 / median_ratio
            if median_ratio != 0 and pd.notna(median_ratio)
            else 1.0
        )

        ratio_cv = (
            ratio_new_over_base.std() / ratio_new_over_base.mean()
            if ratio_new_over_base.mean() != 0
            else np.nan
        )

        method = "overlap_ratio"

    else:
        median_ratio = np.nan
        ratio_cv = np.nan
        scale_new_to_base = 1.0
        method = "no_rescale_insufficient_overlap"

    return {
        "overlap_days": len(overlap),
        "valid_ratio_days": len(valid),
        "overlap_corr": corr,
        "median_ratio_new_over_base": median_ratio,
        "scale_new_to_base": scale_new_to_base,
        "ratio_cv": ratio_cv,
        "method": method,
    }


def stitch_chunks_overlap_ratio(
    chunk_files: list[Path],
    query: str,
    min_overlap_days: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    chunk_files = sorted(chunk_files)

    if not chunk_files:
        return (
            pd.DataFrame(columns=["Time", query]),
            pd.DataFrame()
        )

    stitched = read_chunk_file(
        chunk_files[0],
        query
    )

    diag_rows = []

    diag_rows.append({
        "step": 0,
        "file": chunk_files[0].name,
        "overlap_days": np.nan,
        "valid_ratio_days": np.nan,
        "overlap_corr": np.nan,
        "median_ratio_new_over_base": np.nan,
        "scale_new_to_base": 1.0,
        "ratio_cv": np.nan,
        "method": "base_chunk",
    })

    for i, path in enumerate(chunk_files[1:], start=1):

        new = read_chunk_file(
            path,
            query
        )

        scale_info = estimate_overlap_scale(
            base=stitched,
            new=new,
            query=query,
            min_overlap_days=min_overlap_days,
        )

        new[query] = (
            new[query]
            * scale_info["scale_new_to_base"]
        )

        combined = pd.concat(
            [stitched, new],
            ignore_index=True
        )

        stitched = (
            combined
            .groupby("Time", as_index=False)[query]
            .mean()
            .sort_values("Time")
            .reset_index(drop=True)
        )

        diag_rows.append({
            "step": i,
            "file": path.name,
            **scale_info,
        })

    stitch_scale_diag = pd.DataFrame(diag_rows)

    return stitched, stitch_scale_diag


def make_stitching_diagnostics(
    stitched: pd.DataFrame,
    query: str
) -> pd.DataFrame:

    s = stitched[query]

    return pd.DataFrame([{
        "query": query,
        "n_rows": len(stitched),
        "min_date": stitched["Time"].min(),
        "max_date": stitched["Time"].max(),
        "missing_values": s.isna().sum(),
        "min_value": s.min(),
        "max_value": s.max(),
        "mean_value": s.mean(),
        "std_value": s.std(),
        "n_zero_values": (s == 0).sum(),
    }])


def fix_one_query_folder(
    query_folder: Path,
    request_counter: int = 0,
) -> int:

    folder = query_folder.name
    query = folder

    chunks_dir = query_folder / "chunks"
    diagnostics_dir = query_folder / "diagnostics"
    stitched_dir = query_folder / "stitched"

    diagnostics_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    stitched_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    chunk_files = sorted(
        chunks_dir.glob("gt_*.csv")
    )

    stitched, scale_diag = stitch_chunks_overlap_ratio(
        chunk_files=chunk_files,
        query=query,
        min_overlap_days=10,
    )

    stitched_path = (
        stitched_dir /
        f"gt_fixed_{folder}_{START_DATE}_{END_DATE}.csv"
    )

    stitched.to_csv(
        stitched_path,
        index=False
    )

    stitch_diag = make_stitching_diagnostics(
        stitched,
        query
    )

    stitch_diag.to_csv(
        diagnostics_dir / "stitching_fixed.csv",
        index=False
    )

    scale_diag.to_csv(
        diagnostics_dir / "overlap_scale_fixed.csv",
        index=False
    )

    print(f"saved: {stitched_path}")

    return request_counter


request_counter = 0

for query_folder in sorted(DATA_DIR.iterdir()):

    if not query_folder.is_dir():
        continue

    if not (query_folder / "chunks").exists():
        continue

    request_counter = fix_one_query_folder(
        query_folder=query_folder,
        request_counter=request_counter,
    )

saved: C:\Python\CSUREMM\data_events\2016_election_results\stitched\gt_fixed_2016_election_results_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2020_election_map\stitched\gt_fixed_2020_election_map_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2020_electoral_map\stitched\gt_fixed_2020_electoral_map_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2022_election\stitched\gt_fixed_2022_election_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2024_election\stitched\gt_fixed_2024_election_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2024_united_states_elections\stitched\gt_fixed_2024_united_states_elections_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2024_united_states_presidential_election\stitched\gt_fixed_2024_united_states_presidential_election_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2024_united_states_presidential_election_in_georgia\stitched\gt_fixed_2024_united_

## Fixing stitch: gt_fix2 -- avoiding scaling with spike ratios in the overlapping window.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


DATA_DIR = Path(r"C:/Python/CSUREMM/data_events")
START_DATE = "2022-01-01"
END_DATE = "2026-05-31"


def read_chunk_file(path: Path, query: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        parse_dates=["Time"]
    )

    value_cols = [
        c for c in df.columns
        if c != "Time"
    ]

    if len(value_cols) != 1:
        raise ValueError(
            f"{path.name} has {len(value_cols)} value columns: {value_cols}"
        )

    value_col = value_cols[0]

    df = (
        df[["Time", value_col]]
        .rename(columns={value_col: query})
        .sort_values("Time")
        .drop_duplicates("Time", keep="first")
        .reset_index(drop=True)
    )

    df[query] = pd.to_numeric(
        df[query],
        errors="coerce"
    )

    return df


def estimate_overlap_scale(
    base: pd.DataFrame,
    new: pd.DataFrame,
    query: str,
    min_overlap_days: int = 10,
    spike_pctile: float = 95.0,
    min_reasonable_scale: float = 0.2,
    max_reasonable_scale: float = 5.0,
    max_ratio_cv: float = 1.0,
    min_overlap_corr: float = 0.3,
) -> dict:

    overlap = pd.merge(
        base[["Time", query]].rename(columns={query: "base_value"}),
        new[["Time", query]].rename(columns={query: "new_value"}),
        on="Time",
        how="inner"
    ).dropna(
        subset=["base_value", "new_value"]
    )

    valid = overlap[
        (overlap["base_value"] > 0) &
        (overlap["new_value"] > 0)
    ].copy()

    valid_before_spike_filter = len(valid)

    if len(valid) >= min_overlap_days:
        base_thresh = valid["base_value"].quantile(spike_pctile / 100)
        new_thresh = valid["new_value"].quantile(spike_pctile / 100)

        calm = valid[
            (valid["base_value"] <= base_thresh) &
            (valid["new_value"] <= new_thresh)
        ]

        if len(calm) >= min_overlap_days:
            valid_for_ratio = calm
            spike_filter_used = True
        else:
            valid_for_ratio = valid
            spike_filter_used = False
    else:
        valid_for_ratio = valid
        spike_filter_used = False

    if (
        len(overlap) >= 2 and
        overlap["base_value"].std() > 0 and
        overlap["new_value"].std() > 0
    ):
        corr = overlap["base_value"].corr(overlap["new_value"])
    else:
        corr = np.nan

    if len(valid_for_ratio) >= min_overlap_days:
        ratio_new_over_base = (
            valid_for_ratio["new_value"] /
            valid_for_ratio["base_value"]
        )

        median_ratio = ratio_new_over_base.median()

        scale_new_to_base = (
            1 / median_ratio
            if median_ratio != 0 and pd.notna(median_ratio)
            else 1.0
        )

        ratio_mean = ratio_new_over_base.mean()
        ratio_cv = (
            ratio_new_over_base.std() / ratio_mean
            if ratio_mean != 0 and pd.notna(ratio_mean)
            else np.nan
        )

        method = (
            "overlap_ratio_spike_filtered"
            if spike_filter_used
            else "overlap_ratio"
        )

    else:
        median_ratio = np.nan
        ratio_cv = np.nan
        scale_new_to_base = 1.0
        method = "no_rescale_insufficient_overlap"

    method_notes = []

    scale_flagged = (
        pd.notna(scale_new_to_base) and
        (
            scale_new_to_base < min_reasonable_scale or
            scale_new_to_base > max_reasonable_scale
        )
    )

    ratio_cv_flagged = (
        pd.notna(ratio_cv) and
        ratio_cv > max_ratio_cv
    )

    corr_flagged = (
        pd.notna(corr) and
        corr < min_overlap_corr
    )

    insufficient_overlap_flagged = (
        len(valid_for_ratio) < min_overlap_days
    )

    if scale_flagged:
        method_notes.append("extreme_scale_factor")

    if ratio_cv_flagged:
        method_notes.append("unstable_overlap_ratio")

    if corr_flagged:
        method_notes.append("low_overlap_corr")

    if insufficient_overlap_flagged:
        method_notes.append("insufficient_valid_overlap")

    return {
        "overlap_days": len(overlap),
        "valid_ratio_days_before_spike_filter": valid_before_spike_filter,
        "valid_ratio_days": len(valid_for_ratio),
        "spike_pctile": spike_pctile,
        "spike_filter_used": spike_filter_used,
        "overlap_corr": corr,
        "median_ratio_new_over_base": median_ratio,
        "scale_new_to_base": scale_new_to_base,
        "scale_flagged": scale_flagged,
        "ratio_cv": ratio_cv,
        "ratio_cv_flagged": ratio_cv_flagged,
        "corr_flagged": corr_flagged,
        "insufficient_overlap_flagged": insufficient_overlap_flagged,
        "method": method,
        "method_notes": ";".join(method_notes),
    }


def stitch_chunks_overlap_ratio(
    chunk_files: list[Path],
    query: str,
    min_overlap_days: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    chunk_files = sorted(chunk_files)

    if not chunk_files:
        return (
            pd.DataFrame(columns=["Time", query]),
            pd.DataFrame()
        )

    stitched = read_chunk_file(
        chunk_files[0],
        query
    )

    diag_rows = []

    diag_rows.append({
        "step": 0,
        "file": chunk_files[0].name,
        "overlap_days": np.nan,
        "valid_ratio_days": np.nan,
        "overlap_corr": np.nan,
        "median_ratio_new_over_base": np.nan,
        "scale_new_to_base": 1.0,
        "ratio_cv": np.nan,
        "method": "base_chunk",
    })

    for i, path in enumerate(chunk_files[1:], start=1):

        new = read_chunk_file(
            path,
            query
        )

        scale_info = estimate_overlap_scale(
            base=stitched,
            new=new,
            query=query,
            min_overlap_days=min_overlap_days,
        )

        new[query] = (
            new[query]
            * scale_info["scale_new_to_base"]
        )

        combined = pd.concat(
            [stitched, new],
            ignore_index=True
        )

        stitched = (
            combined
            .groupby("Time", as_index=False)[query]
            .mean()
            .sort_values("Time")
            .reset_index(drop=True)
        )

        diag_rows.append({
            "step": i,
            "file": path.name,
            **scale_info,
        })

    stitch_scale_diag = pd.DataFrame(diag_rows)

    return stitched, stitch_scale_diag


def fix_one_query_folder(
    query_folder: Path,
    request_counter: int = 0,
) -> int:

    folder = query_folder.name
    query = folder

    chunks_dir = query_folder / "chunks"
    diagnostics_dir = query_folder / "diagnostics"
    stitched_dir = query_folder / "stitched"

    diagnostics_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    stitched_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    chunk_files = sorted(
        chunks_dir.glob("gt_*.csv")
    )

    stitched, scale_diag = stitch_chunks_overlap_ratio(
        chunk_files=chunk_files,
        query=query,
        min_overlap_days=10,
    )

    stitched_path = (
        stitched_dir /
        f"gt_fixed02_{folder}_{START_DATE}_{END_DATE}.csv"
    )

    stitched.to_csv(
        stitched_path,
        index=False
    )

    print(f"saved: {stitched_path}")

    return request_counter


# -----------------------------------------------------------------------------
# Stitch all queries and aggregate diagnostics
# -----------------------------------------------------------------------------

request_counter = 0
all_scale_diagnostics = []

for query_folder in sorted(DATA_DIR.iterdir()):

    if not query_folder.is_dir():
        continue

    if not (query_folder / "chunks").exists():
        continue

    folder = query_folder.name

    request_counter = fix_one_query_folder(
        query_folder=query_folder,
        request_counter=request_counter,
    )

    # Read the diagnostics returned during stitching
    _, scale_diag = stitch_chunks_overlap_ratio(
        chunk_files=sorted((query_folder / "chunks").glob("gt_*.csv")),
        query=folder,
        min_overlap_days=10,
    )

    scale_diag = scale_diag.copy()
    scale_diag.insert(0, "query", folder)

    all_scale_diagnostics.append(scale_diag)

# -----------------------------------------------------------------------------
# Save one master diagnostics table
# -----------------------------------------------------------------------------

if all_scale_diagnostics:

    master_scale_diag = (
        pd.concat(
            all_scale_diagnostics,
            ignore_index=True
        )
        .sort_values(
            ["query", "step"]
        )
        .reset_index(drop=True)
    )

    master_path = (
        DATA_DIR /
        "scale_diagnostics.csv"
    )

    master_scale_diag.to_csv(
        master_path,
        index=False
    )

    print(f"saved: {master_path}")

saved: C:\Python\CSUREMM\data_events\2016_election_results\stitched\gt_fixed02_2016_election_results_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2020_election_map\stitched\gt_fixed02_2020_election_map_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2020_election_results\stitched\gt_fixed02_2020_election_results_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2020_electoral_map\stitched\gt_fixed02_2020_electoral_map_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2022_election\stitched\gt_fixed02_2022_election_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2022_election_results\stitched\gt_fixed02_2022_election_results_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2024_election\stitched\gt_fixed02_2024_election_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_events\2024_election_results\stitched\gt_fixed02_2024_election_results_2022-01-01_2026-05-31.csv
saved: C:\Python\CSUREMM\data_

In [6]:
# fixing temporary stitch for primitive words

from pathlib import Path
import numpy as np
import pandas as pd


DATA_DIR = Path(r"C:/Python/CSUREMM/dictionary V0.0/data_primitive")
START_DATE = "2022-01-01"
END_DATE = "2026-05-31"


def read_chunk_file(path: Path, query: str) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        parse_dates=["Time"]
    )

    value_cols = [
        c for c in df.columns
        if c != "Time"
    ]

    if len(value_cols) != 1:
        raise ValueError(
            f"{path.name} has {len(value_cols)} value columns: {value_cols}"
        )

    value_col = value_cols[0]

    df = (
        df[["Time", value_col]]
        .rename(columns={value_col: query})
        .sort_values("Time")
        .drop_duplicates("Time", keep="first")
        .reset_index(drop=True)
    )

    df[query] = pd.to_numeric(
        df[query],
        errors="coerce"
    )

    return df


def estimate_overlap_scale(
    base: pd.DataFrame,
    new: pd.DataFrame,
    query: str,
    min_overlap_days: int = 10,
) -> dict:

    overlap = pd.merge(
        base[["Time", query]].rename(columns={query: "base_value"}),
        new[["Time", query]].rename(columns={query: "new_value"}),
        on="Time",
        how="inner"
    ).dropna(
        subset=["base_value", "new_value"]
    )

    valid = overlap[
        (overlap["base_value"] > 0) &
        (overlap["new_value"] > 0)
    ].copy()

    if len(overlap) >= 2 and overlap["base_value"].std() > 0 and overlap["new_value"].std() > 0:
        corr = overlap["base_value"].corr(overlap["new_value"])
    else:
        corr = np.nan

    if len(valid) >= min_overlap_days:
        ratio_new_over_base = (
            valid["new_value"] /
            valid["base_value"]
        )

        median_ratio = ratio_new_over_base.median()

        scale_new_to_base = (
            1 / median_ratio
            if median_ratio != 0 and pd.notna(median_ratio)
            else 1.0
        )

        ratio_cv = (
            ratio_new_over_base.std() / ratio_new_over_base.mean()
            if ratio_new_over_base.mean() != 0
            else np.nan
        )

        method = "overlap_ratio"

    else:
        median_ratio = np.nan
        ratio_cv = np.nan
        scale_new_to_base = 1.0
        method = "no_rescale_insufficient_overlap"

    return {
        "overlap_days": len(overlap),
        "valid_ratio_days": len(valid),
        "overlap_corr": corr,
        "median_ratio_new_over_base": median_ratio,
        "scale_new_to_base": scale_new_to_base,
        "ratio_cv": ratio_cv,
        "method": method,
    }


def stitch_chunks_overlap_ratio(
    chunk_files: list[Path],
    query: str,
    min_overlap_days: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    chunk_files = sorted(chunk_files)

    if not chunk_files:
        return (
            pd.DataFrame(columns=["Time", query]),
            pd.DataFrame()
        )

    stitched = read_chunk_file(
        chunk_files[0],
        query
    )

    diag_rows = []

    diag_rows.append({
        "step": 0,
        "file": chunk_files[0].name,
        "overlap_days": np.nan,
        "valid_ratio_days": np.nan,
        "overlap_corr": np.nan,
        "median_ratio_new_over_base": np.nan,
        "scale_new_to_base": 1.0,
        "ratio_cv": np.nan,
        "method": "base_chunk",
    })

    for i, path in enumerate(chunk_files[1:], start=1):

        new = read_chunk_file(
            path,
            query
        )

        scale_info = estimate_overlap_scale(
            base=stitched,
            new=new,
            query=query,
            min_overlap_days=min_overlap_days,
        )

        new[query] = (
            new[query]
            * scale_info["scale_new_to_base"]
        )

        combined = pd.concat(
            [stitched, new],
            ignore_index=True
        )

        stitched = (
            combined
            .groupby("Time", as_index=False)[query]
            .mean()
            .sort_values("Time")
            .reset_index(drop=True)
        )

        diag_rows.append({
            "step": i,
            "file": path.name,
            **scale_info,
        })

    stitch_scale_diag = pd.DataFrame(diag_rows)

    return stitched, stitch_scale_diag


def make_stitching_diagnostics(
    stitched: pd.DataFrame,
    query: str
) -> pd.DataFrame:

    s = stitched[query]

    return pd.DataFrame([{
        "query": query,
        "n_rows": len(stitched),
        "min_date": stitched["Time"].min(),
        "max_date": stitched["Time"].max(),
        "missing_values": s.isna().sum(),
        "min_value": s.min(),
        "max_value": s.max(),
        "mean_value": s.mean(),
        "std_value": s.std(),
        "n_zero_values": (s == 0).sum(),
    }])


def fix_one_query_folder(query_folder: Path):

    folder = query_folder.name
    query = folder

    stitched_dir = Path(
        r"C:/Python/CSUREMM/dictionary V0.0/data_primitive_events"
    )
    stitched_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    chunk_files = sorted(
        query_folder.glob("gt_daily_*.csv")
    )

    if len(chunk_files) == 0:
        print(f"Skipping {folder}: no gt_daily_*.csv files found.")
        return

    stitched, _ = stitch_chunks_overlap_ratio(
        chunk_files=chunk_files,
        query=query,
        min_overlap_days=10,
    )

    stitched.to_csv(
        stitched_dir / f"gt_fixed_{folder}.csv",
        index=False
    )

    print(f"Saved {folder}: {len(stitched)} rows")


for query_folder in sorted(DATA_DIR.iterdir()):

    if not query_folder.is_dir():
        continue

    fix_one_query_folder(query_folder)

Saved accrue: 1612 rows
Saved advantage: 1612 rows
Saved affluence: 1612 rows
Saved affluent: 1612 rows
Saved afloat: 1612 rows
Saved allowance: 1612 rows
Saved aristocracy: 1612 rows
Saved aristocrat: 1612 rows
Saved aristocratic: 1612 rows
Saved associate: 1612 rows
Saved backer: 1612 rows
Saved backward: 1612 rows
Saved backwardness: 1612 rows
Saved bankrupt: 1612 rows
Saved bankruptcy: 1612 rows
Saved bargain: 1612 rows
Saved beggar: 1612 rows
Saved benefactor: 1612 rows
Saved beneficiary: 1612 rows
Saved benefit: 1612 rows
Saved benevolence: 1612 rows
Saved benevolent: 1612 rows
Saved bequeath: 1612 rows
Saved betroth: 1612 rows
Saved betrothal: 1612 rows
Saved blackmail: 1612 rows
Saved bonus: 1612 rows
Saved boom: 1612 rows
Saved breadwinner: 1612 rows
Saved bribe: 1612 rows
Saved broke: 1612 rows
Saved bum: 1612 rows
Saved buy: 1612 rows
Saved capitalize: 1612 rows
Saved charitable: 1612 rows
Saved charity: 1612 rows
Saved cheap: 1612 rows
Saved colony: 1612 rows
Saved commoner

In [10]:
from pathlib import Path
import shutil


PRIMITIVE_EVENTS_DIR = Path(
    r"C:/Python/CSUREMM/dictionary V0.0/data_primitive_for_events"
)

DATA_EVENTS_DIR = Path(
    r"C:/Python/CSUREMM/data_events"
)

MASTER_DIR = Path(
    r"C:/Python/CSUREMM/data_primitve_events"
)

MASTER_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# 1. Copy all fixed primitive files
for file in sorted(PRIMITIVE_EVENTS_DIR.glob("gt_fixed_*.csv")):

    dest = MASTER_DIR / file.name

    shutil.copy2(
        file,
        dest
    )

    print(f"Copied primitive fixed: {file.name}")


# 2. Copy all stitched event files
for query_folder in sorted(DATA_EVENTS_DIR.iterdir()):

    if not query_folder.is_dir():
        continue

    stitched_dir = query_folder / "stitched"

    if not stitched_dir.exists():
        continue

    stitched_files = sorted(
        stitched_dir.glob("gt_fixed_*.csv")
    )

    if len(stitched_files) == 0:
        continue

    if len(stitched_files) > 1:
        print(
            f"Warning: multiple stitched files in {query_folder.name}; using first."
        )

    file = stitched_files[0]

    dest = MASTER_DIR / file.name

    shutil.copy2(
        file,
        dest
    )

    print(f"Copied event stitched: {file.name}")


print("Done.")
print(f"Master folder: {MASTER_DIR}")

Copied primitive fixed: gt_fixed_accrue.csv
Copied primitive fixed: gt_fixed_advantage.csv
Copied primitive fixed: gt_fixed_affluence.csv
Copied primitive fixed: gt_fixed_affluent.csv
Copied primitive fixed: gt_fixed_afloat.csv
Copied primitive fixed: gt_fixed_allowance.csv
Copied primitive fixed: gt_fixed_aristocracy.csv
Copied primitive fixed: gt_fixed_aristocrat.csv
Copied primitive fixed: gt_fixed_aristocratic.csv
Copied primitive fixed: gt_fixed_associate.csv
Copied primitive fixed: gt_fixed_backer.csv
Copied primitive fixed: gt_fixed_backward.csv
Copied primitive fixed: gt_fixed_backwardness.csv
Copied primitive fixed: gt_fixed_bankrupt.csv
Copied primitive fixed: gt_fixed_bankruptcy.csv
Copied primitive fixed: gt_fixed_bargain.csv
Copied primitive fixed: gt_fixed_beggar.csv
Copied primitive fixed: gt_fixed_benefactor.csv
Copied primitive fixed: gt_fixed_beneficiary.csv
Copied primitive fixed: gt_fixed_benefit.csv
Copied primitive fixed: gt_fixed_benevolence.csv
Copied primitive 